# 02 Analysis: Track 2, Cardiovascular Health in Women

This notebook follows steps 3 to 7 of the [workflow](../../3-workflow.md): explore, build a model, check it, interpret and make a headline visual.

Run `01_cleaning.ipynb` first. Settings marked **You decide** start as `None`.

In [ ]:
# Setup: run this cell first. It finds the track folder and the data.
import re, sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PUBLIC_REPO = "https://github.com/Harvard-Undergraduate-OpenBio/openbio-datathon-2026.git"
IN_COLAB = "google.colab" in sys.modules

def find_track_dir():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "track-02" / "data").is_dir():
            return folder / "track-02"
        if folder.name == "track-02" and (folder / "data").is_dir():
            return folder
    if IN_COLAB:
        dest = Path("/content/openbio-datathon-2026")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", PUBLIC_REPO, str(dest)], check=True)
        return dest / "track-02"
    return None

TRACK = find_track_dir()
TEAM = TRACK / "team" if TRACK else Path.cwd()
CLEAN = TEAM / "data_clean"
FIGURES = TEAM / "figures"
for folder in (CLEAN, FIGURES, TEAM / "docs"):
    folder.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [d for d in [TRACK / "data" if TRACK else None, TRACK / "data" / "gbd" if TRACK else None,
               Path.cwd(), Path("/content"), Path("/content/drive/MyDrive/datathon")] if d]

def missing(name):
    print(f"Fill in {name} in the cell above, then rerun this cell.")

print("Track folder:", TRACK)
print("Team folder: ", TEAM)
if IN_COLAB:
    print("On Colab, outputs are saved in this session only. Download them or copy them into your team repository.")


## Step 1: Your question

**You decide:**

- `QUESTION`: one sentence, also written in your team README.
- `DATASET`: `"d1"`, `"d2"`, `"s1"`, `"s2"` or `"s2_long"`.
- `OUTCOME`: the column you want to explain.
- `PREDICTORS`: a list of columns. Wrap ordered or text categories in `C()`, for example `"C(cholesterol_level)"`.
- `MODEL`: `"logistic"` for a yes/no outcome or `"linear"` for a number.

In [ ]:
QUESTION = None
DATASET = None
OUTCOME = None
PREDICTORS = None
MODEL = None

READY = all(v is not None for v in (QUESTION, DATASET, OUTCOME, PREDICTORS, MODEL))
if not READY:
    print("Fill in QUESTION, DATASET, OUTCOME, PREDICTORS and MODEL, then rerun the notebook.")
    print("Example: DATASET='d1', OUTCOME='cardio', PREDICTORS=['age_years', 'C(cholesterol_level)'], MODEL='logistic'")


In [ ]:
df = None
if READY:
    path = CLEAN / f"{DATASET}_clean.csv"
    if not path.exists():
        print(f"{path.name} not found. Run 01_cleaning.ipynb first.")
        READY = False
    else:
        df = pd.read_csv(path)
        for col in [c for c in df.columns if c.endswith("_level")]:
            order = {"cholesterol_level": ["normal", "above normal", "well above normal"],
                     "gluc_level": ["normal", "above normal", "well above normal"],
                     "alcohol_level": ["None", "Light", "Moderate", "Heavy"],
                     "diet_level": ["Unhealthy", "Average", "Healthy"]}.get(col)
            if order:
                df[col] = pd.Categorical(df[col], categories=order, ordered=True)
        print(QUESTION)
        print(f"Loaded {len(df):,} rows from {path.name}")


## Step 2: Explore

Sample size, summary statistics and each predictor plotted against the outcome. Note anything surprising.

In [ ]:
def base_column(term):
    m = re.match(r"C\((\w+)", term)
    return m.group(1) if m else term

def nice_term(term):
    """'C(cholesterol_level)[T.above normal]' -> 'cholesterol_level: above normal'."""
    m = re.match(r"C\((\w+)[^)]*\)\[T\.(.+)\]", term)
    return f"{m.group(1)}: {m.group(2)}" if m else term

if READY:
    cols = [OUTCOME] + [base_column(p) for p in PREDICTORS]
    data = df[cols].dropna()
    print(f"Rows with no missing values in these columns: {len(data):,} of {len(df):,}")
    display(data.describe(include="all").round(2))

    fig, axes = plt.subplots(1, len(PREDICTORS), figsize=(5 * len(PREDICTORS), 3.5), squeeze=False)
    for ax, term in zip(axes[0], PREDICTORS):
        col = base_column(term)
        if term.startswith("C(") or data[col].nunique() <= 10:
            data.groupby(col, observed=True)[OUTCOME].mean().plot.bar(ax=ax, rot=0)
            ax.set_ylabel(f"Mean {OUTCOME}")
        elif MODEL == "logistic":
            binned = pd.qcut(data[col], 10, duplicates="drop")
            share = data.groupby(binned, observed=True)[OUTCOME].mean()
            ax.plot([b.mid for b in share.index], share.values, marker="o")
            ax.set_ylabel(f"Share with {OUTCOME} = 1")
        else:
            sample = data.sample(min(len(data), 3000), random_state=0)
            ax.scatter(sample[col], sample[OUTCOME], s=5, alpha=0.3)
            ax.set_ylabel(OUTCOME)
        ax.set_xlabel(col)
    plt.tight_layout()
    plt.show()


## Step 3: Build the model

- **Logistic:** prints odds ratios (OR) with 95% confidence intervals and draws a forest plot. OR = 1 means no association.
- **Linear:** prints the slope and R². If you have one numeric predictor, it draws the fitted line with a confidence band.

In [ ]:
import statsmodels.formula.api as smf

result = None
if READY:
    formula = f"{OUTCOME} ~ " + " + ".join(PREDICTORS)
    print("Formula:", formula)
    if MODEL == "logistic":
        result = smf.logit(formula, data=data).fit(disp=False)
        table = pd.DataFrame({"OR": np.exp(result.params),
                              "CI low": np.exp(result.conf_int()[0]),
                              "CI high": np.exp(result.conf_int()[1]),
                              "p-value": result.pvalues}).drop(index="Intercept")
        table.index = [nice_term(i) for i in table.index]
        display(table.round(3))

        fig, ax = plt.subplots(figsize=(7, 0.5 * len(table) + 1.5))
        y = np.arange(len(table))[::-1]
        ax.errorbar(table["OR"], y, xerr=[table["OR"] - table["CI low"], table["CI high"] - table["OR"]],
                    fmt="o", capsize=4)
        ax.axvline(1, color="grey", linestyle="--")
        ax.set_yticks(y, table.index)
        ax.set_xlabel("Odds ratio (95% CI)")
        ax.set_title("Odds ratios")
        plt.tight_layout()
        fig.savefig(FIGURES / "forest_plot.png", dpi=200)
        plt.show()
    elif MODEL == "linear":
        result = smf.ols(formula, data=data).fit()
        table = pd.DataFrame({"Slope": result.params, "CI low": result.conf_int()[0],
                              "CI high": result.conf_int()[1], "p-value": result.pvalues})
        table.index = [nice_term(i) for i in table.index]
        display(table.round(3))
        print(f"R-squared: {result.rsquared:.3f}   n = {int(result.nobs):,}")

        if len(PREDICTORS) == 1 and not PREDICTORS[0].startswith("C("):
            x = PREDICTORS[0]
            grid = pd.DataFrame({x: np.linspace(data[x].min(), data[x].max(), 100)})
            band = result.get_prediction(grid).summary_frame(alpha=0.05)
            sample = data.sample(min(len(data), 3000), random_state=0)
            fig, ax = plt.subplots(figsize=(7, 4.5))
            ax.scatter(sample[x], sample[OUTCOME], s=6, alpha=0.3)
            ax.plot(grid[x], band["mean"], color="C1")
            ax.fill_between(grid[x], band["mean_ci_lower"], band["mean_ci_upper"], color="C1", alpha=0.3)
            ax.set_xlabel(x)
            ax.set_ylabel(OUTCOME)
            ax.set_title(f"Slope {result.params[x]:.2f}, R\u00b2 {result.rsquared:.2f}")
            plt.tight_layout()
            fig.savefig(FIGURES / "regression_line.png", dpi=200)
            plt.show()
    else:
        print('MODEL must be "logistic" or "linear".')


## Step 4: Check your work

- **Logistic:** fits on 80% of the rows and reports the AUC on the other 20%. 0.5 is a coin flip, 1 is perfect.
- **Linear:** plots residuals against fitted values. A clear curve or funnel shape means a straight line is a poor fit.

Then change one decision (a cleaning range or a predictor) and rerun to see whether your main result holds.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, RocCurveDisplay

if READY and result is not None:
    if MODEL == "logistic":
        train, test = train_test_split(data, test_size=0.2, random_state=42, stratify=data[OUTCOME])
        holdout = smf.logit(formula, data=train).fit(disp=False)
        prob = holdout.predict(test)
        auc = roc_auc_score(test[OUTCOME], prob)
        print(f"Held-out AUC: {auc:.3f} (train n = {len(train):,}, test n = {len(test):,})")
        RocCurveDisplay.from_predictions(test[OUTCOME], prob)
        plt.plot([0, 1], [0, 1], "--", color="grey")
        plt.show()
    else:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.scatter(result.fittedvalues, result.resid, s=5, alpha=0.3)
        ax.axhline(0, color="grey", linestyle="--")
        ax.set_xlabel("Fitted value")
        ax.set_ylabel("Residual")
        plt.tight_layout()
        plt.show()


## Step 5: Interpret

Write your main result as one plain sentence with the effect size and its confidence interval, then check it against the [ground rules](../../README.md#ground-rules). Say "associated with," not "causes."

**Your sentence:** (write it here)

## Step 6: Headline visual

One graph for a non-specialist, used in both the video and the article.

**You decide:** a title that states the finding in plain words, and everyday axis labels with units. `HEADLINE_TERMS` picks which model terms to show (logistic) and `LABELS` renames them.

In [ ]:
HEADLINE_TITLE = None    # e.g. "Higher cholesterol is linked to higher odds of heart disease in women"
X_LABEL = None
Y_LABEL = None
HEADLINE_TERMS = None    # logistic only: list of terms from the OR table, or None for all
LABELS = {}              # e.g. {"age_years": "Age (per year)"}

if READY and result is not None:
    if HEADLINE_TITLE is None:
        missing("HEADLINE_TITLE")
    fig, ax = plt.subplots(figsize=(8, 5))
    if MODEL == "logistic":
        t = table if HEADLINE_TERMS is None else table.loc[HEADLINE_TERMS]
        fig.set_size_inches(8, max(3, 1.5 + 0.7 * len(t)))
        y = np.arange(len(t))[::-1]
        ax.errorbar(t["OR"], y, xerr=[t["OR"] - t["CI low"], t["CI high"] - t["OR"]],
                    fmt="o", markersize=8, capsize=5, color="#0b7a5e")
        ax.axvline(1, color="grey", linestyle="--")
        ax.set_yticks(y, [LABELS.get(i, i) for i in t.index])
        ax.set_xlabel(X_LABEL or "Odds ratio (95% confidence interval)")
    elif len(PREDICTORS) == 1 and not PREDICTORS[0].startswith("C("):
        ax.scatter(sample[x], sample[OUTCOME], s=6, alpha=0.2, color="grey")
        ax.plot(grid[x], band["mean"], color="#0b7a5e", linewidth=2.5)
        ax.fill_between(grid[x], band["mean_ci_lower"], band["mean_ci_upper"], color="#0b7a5e", alpha=0.25)
        ax.set_xlabel(X_LABEL or x)
        ax.set_ylabel(Y_LABEL or OUTCOME)
    else:
        t = table.drop(index="Intercept")
        fig.set_size_inches(8, max(3, 1.5 + 0.7 * len(t)))
        y = np.arange(len(t))[::-1]
        ax.errorbar(t["Slope"], y, xerr=[t["Slope"] - t["CI low"], t["CI high"] - t["Slope"]],
                    fmt="o", markersize=8, capsize=5, color="#0b7a5e")
        ax.axvline(0, color="grey", linestyle="--")
        ax.set_yticks(y, [LABELS.get(i, i) for i in t.index])
        ax.set_xlabel(X_LABEL or f"Change in {OUTCOME} per unit (95% confidence interval)")
    ax.set_title(HEADLINE_TITLE or "Add a title that states your finding", fontsize=13)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    plt.tight_layout()
    fig.savefig(FIGURES / "headline.png", dpi=300)
    plt.show()
    print("Saved", FIGURES / "headline.png")


---

This notebook is part of a student data analysis. It is not medical advice, a diagnostic tool or clinical guidance.